# **Introduction to Customer Churn Prediction**

Customer Churn - defined as when customers or subscribers discontinue doing business with a firm or service

- Telecom business has an annual churn rate of 15-25 percent in this highly competitive market
- to give time to every customer, instead predicting high risk customer through churn predictions and only focusing on them


Objective 

- What's the % of Churn Customers and customers that keep in with the active services?
- Is there any patterns in Churn Customers based on the gender?
- Is there any patterns/preference in Churn Customers based on the type of service provided?
- What's the most profitable service types?
- Which features and services are most profitable?
- Many more questions that will arise during the analysis

# Dataset

- RowNumber: corresponds to the record (row) number and has no effect on the output.
- CustomerId: contains random values and has no effect on customer leaving the bank.
- Surname: the surname of a customer has no impact on their decision to leave the bank.
- CreditScore: can have an effect on customer churn, since a customer with a higher credit score is less likely to leave the bank.
- Geography: a customer’s location can affect their decision to leave the bank.
- Gender: it’s interesting to explore whether gender plays a role in a customer leaving the bank.
- Age: this is certainly relevant, since older customers are less likely to leave their bank than younger ones.
- Tenure: refers to the number of years that the customer has been a client of the bank. Normally, older clients are more loyal and less likely to leave a bank.
- Balance: also a very good indicator of customer churn, as people with a higher balance in their accounts are less likely to leave the bank compared to those with lower balances.
- NumOfProducts: refers to the number of products that a customer has purchased through the bank.
- HasCrCard: denotes whether or not a customer has a credit card. This column is also relevant, since people with a credit card are less likely to leave the bank.
- IsActiveMember: active customers are less likely to leave the bank.
- EstimatedSalary: as with balance, people with lower salaries are more likely to leave the bank compared to those with higher salaries.
- Exited: whether or not the customer left the bank. (0=No,1=Yes)

# Loading necessary libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as pyplot
from sklearn.model_selection import cross_val_predict # trains model on K-1 fold, eg ([1, 0, 1, 1] as it gives prediction not score. )
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
# confusion matrix - grid comparing actual and predicted values (TP, TN, FP (type I error), FN (type II error))
# classification metrics - numbers from confusion matrix converting them into ratio (0.0 to 1.0)
# precision_score - accuracy of +ve predictions (of how many times model said '+ve' it was actually +ve)
# recall_score - coverage of actual +ve (of all the actual +ve, how many model find out)
# f1_score - how to balance recall and precision
# roc_score - a plot of true postive rate (recall) vs false positive rate at every possible prob. threshold
# roc_auc_score - a single number representing the probability that the model will rank a random +ve instance higher than a random negative one
from sklearn.linear_model import LogisticRegression # powerful for classification task, steps to work on it - import -> instantiate -> fit -> predict
from sklearn.neighbors import KNeighborsClassifier # supervised ml algo, classify new k data point on basis of major vote system to find most suitable group of neighbors
from sklearn.svm import SVC # finding an optimal hyperplane to separate different classes in a feature space 
from catboost import CatBoostClassifier # automatic advanced encoding sceheme to convert text (like )
from sklearn.ensemble import GradientBoostingClassifier # classfication where goal is to predict a discrete category or class label for new data points.
from sklearn.tree import DecisionTreeClassifier # split data into branches based on feature values to create the most 'pure' groups possible
from sklearn.ensemble import RandomForestClassifier # to predict a categorical outcome
from lightgbm import LGBMClassifier # (light gradient boosting machine) used for building tree based ensemble models
from sklearn.model_selection import train_test_split # used to divide dataset into train and test data usually 80 - 20 ratio
from sklearn import preprocessing # transforming raw data to clean structured version of it
from sklearn.metrics import accuracy_score, recall_score 
# accuracy score - overall correctness ((TP + TN) / (TP + TN + FN + FP))
# recall score - finding all positives (TP / TP + FN)
from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score, GridSearchCV

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 
warnings.filterwarnings("ignore", category=UserWarning) 

%config InlineBackend.figure_format = 'retina'

# to display all columns and rows:
pd.set_option('display.max_columns', None); pd.set_option('display.max_rows', None);

In [2]:
df = pd.read_csv('/kaggle/input/datasets/prashantycode/churn-dataset-ml/churn.csv.xls')

1) EDA

In [3]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
df.shape

(10000, 14)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [6]:
df.describe()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [7]:
# Categorical Variables

categorical_variables = [col for col in df.columns if col in '0'
                        or df[col].nunique() <= 11
                        and col not in 'Exited']

categorical_variables

['Geography',
 'Gender',
 'Tenure',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember']

In [8]:
# Numeric Variables

numeric_variables = [col for col in df.columns if df[col].dtype != 'object'
                    and df[col].nunique() > 11
                    and col not in 'CustomerId']

numeric_variables

['RowNumber', 'CreditScore', 'Age', 'Balance', 'EstimatedSalary']

Exited (Dependent Variable)

In [9]:
# Frequency of classes of dependent variable

df['Exited'].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [10]:
# Customer leaving the bank

churn = df.loc[df['Exited'] == 1]

In [11]:
# Customer who did not leave the bank

not_churn = df.loc[df['Exited'] == 0]